# Synthetic Data Generator for MongoDB Cluster

This notebook generates realistic synthetic data (customers, products, orders, sensor telemetry) and bulk-inserts it into the multi-zone MongoDB cluster provisioned by this repository.

### Prerequisites:
- Running MongoDB cluster (deployed via `make deploy` in this repository).
- If running in GCP Colab Enterprise, ensure this runtime is deployed within the `mongodb-network` VPC and subnet.

In [ ]:
# 1. Install required dependencies
!pip install --quiet pymongo faker tqdm

## 2. Configuration & Connection Setup

In [ ]:
import os
import random
import time
from datetime import datetime, timedelta
from typing import List, Dict, Any

from pymongo import MongoClient, errors
from faker import Faker

# MongoDB Cluster Connection Parameters
SEED_IP = os.getenv("MONGODB_SEED_IP", "10.42.0.2")
NODE_IPS = os.getenv("MONGODB_NODE_IPS", "10.42.0.2,10.42.0.3,10.42.0.4").split(",")
PORT = int(os.getenv("MONGODB_PORT", "27017"))
REPLICA_SET = os.getenv("MONGODB_REPLICA_SET", "rs-analytics")
DB_NAME = os.getenv("MONGODB_DATABASE", "ecommerce_analytics")

# Construct Replica Set Connection URI
hosts = ",".join([f"{ip.strip()}:{PORT}" for ip in NODE_IPS])
MONGO_URI = f"mongodb://{hosts}/?replicaSet={REPLICA_SET}&serverSelectionTimeoutMS=5000&directConnection=false"

print(f"Connecting to MongoDB Replica Set '{REPLICA_SET}'...")
print(f"Target Hosts: {hosts}")

try:
    client = MongoClient(MONGO_URI)
    # Test connection and fetch hello/ismaster status
    client.admin.command('ping')
    hello = client.admin.command('hello')
    print("\n✅ Successfully connected to MongoDB Cluster!")
    print(f"  Primary Node: {hello.get('primary')}")
    print(f"  Cluster Members: {hello.get('hosts')}")
    print(f"  Writable Primary: {hello.get('isWritablePrimary', hello.get('ismaster'))}")
    db = client[DB_NAME]
except Exception as e:
    print(f"\n❌ Failed to connect to MongoDB cluster at {hosts}:")
    print(e)
    print("\n💡 Troubleshooting:")
    print("1. If running in GCP Colab Enterprise, verify the runtime is attached to 'mongodb-network'.")
    print("2. Ensure the MongoDB nodes are active (`make output` to view IPs).")

## 3. Data Generation Parameters

In [ ]:
# Configure document generation volumes
NUM_CUSTOMERS = 2500
NUM_PRODUCTS = 250
NUM_ORDERS = 12000
NUM_SENSOR_LOGS = 20000
BATCH_SIZE = 1000

fake = Faker()
Faker.seed(42)  # Seed for reproducible mock data
random.seed(42)
print(f"Initialized data generator parameters (Batch Size: {BATCH_SIZE}).")

## 4. Synthetic Data Generation Functions

In [ ]:
def generate_customers(count: int) -> List[Dict[str, Any]]:
    customers = []
    tiers = ["standard", "silver", "gold", "platinum"]
    statuses = ["active", "active", "active", "inactive", "suspended"]
    
    for _ in range(count):
        reg_date = fake.date_time_between(start_date="-2y", end_date="now")
        customer = {
            "customer_id": f"CUST-{fake.unique.random_number(digits=8, fix_len=True)}",
            "first_name": fake.first_name(),
            "last_name": fake.last_name(),
            "email": fake.unique.company_email(),
            "phone": fake.phone_number(),
            "account_tier": random.choice(tiers),
            "status": random.choice(statuses),
            "address": {
                "street": fake.street_address(),
                "city": fake.city(),
                "state": fake.state_abbr(),
                "zipcode": fake.zipcode(),
                "country": "USA"
            },
            "created_at": reg_date,
            "updated_at": reg_date + timedelta(days=random.randint(0, 100))
        }
        customers.append(customer)
    return customers

def generate_products(count: int) -> List[Dict[str, Any]]:
    categories = ["Electronics", "Home & Kitchen", "Apparel", "Books", "Sports & Outdoors", "Beauty"]
    products = []
    
    for _ in range(count):
        products.append({
            "product_id": f"PROD-{fake.unique.random_number(digits=6, fix_len=True)}",
            "name": fake.catch_phrase(),
            "category": random.choice(categories),
            "price": round(random.uniform(9.99, 1499.99), 2),
            "stock_quantity": random.randint(0, 500),
            "sku": fake.bothify(text="SKU-????-#####").upper(),
            "created_at": fake.date_time_between(start_date="-3y", end_date="-1y")
        })
    return products

def generate_orders(count: int, customer_ids: List[str], products: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    payment_methods = ["credit_card", "debit_card", "paypal", "apple_pay", "bank_transfer"]
    order_statuses = ["completed", "completed", "completed", "shipped", "processing", "cancelled"]
    orders = []
    
    for _ in range(count):
        cust_id = random.choice(customer_ids)
        num_items = random.randint(1, 5)
        selected_products = random.sample(products, num_items)
        
        items = []
        subtotal = 0.0
        for prod in selected_products:
            qty = random.randint(1, 4)
            unit_price = prod["price"]
            item_subtotal = round(qty * unit_price, 2)
            subtotal += item_subtotal
            items.append({
                "product_id": prod["product_id"],
                "product_name": prod["name"],
                "quantity": qty,
                "unit_price": unit_price,
                "subtotal": item_subtotal
            })
            
        tax = round(subtotal * 0.08, 2)
        shipping = 0.0 if subtotal > 100 else 9.99
        total_amount = round(subtotal + tax + shipping, 2)
        order_date = fake.date_time_between(start_date="-1y", end_date="now")
        
        orders.append({
            "order_id": f"ORD-{fake.unique.random_number(digits=10, fix_len=True)}",
            "customer_id": cust_id,
            "order_date": order_date,
            "status": random.choice(order_statuses),
            "items": items,
            "payment": {
                "method": random.choice(payment_methods),
                "status": "paid" if total_amount > 0 else "refunded",
                "transaction_id": f"TXN-{fake.uuid4()[:8]}"
            },
            "pricing": {
                "subtotal": round(subtotal, 2),
                "tax": tax,
                "shipping": shipping,
                "total_amount": total_amount
            }
        })
    return orders

def generate_sensor_telemetry(count: int) -> List[Dict[str, Any]]:
    locations = ["us-central1-a", "us-central1-b", "us-central1-c"]
    device_types = ["temperature_sensor", "disk_monitor", "network_gateway", "app_server"]
    telemetry = []
    
    for _ in range(count):
        dev_id = f"DEV-{random.randint(100, 150)}"
        ts = fake.date_time_between(start_date="-7d", end_date="now")
        telemetry.append({
            "device_id": dev_id,
            "device_type": random.choice(device_types),
            "location": random.choice(locations),
            "timestamp": ts,
            "metrics": {
                "cpu_utilization_pct": round(random.uniform(5.0, 95.0), 2),
                "memory_used_mb": random.randint(1024, 32768),
                "temperature_celsius": round(random.uniform(30.0, 75.0), 1),
                "disk_iops": random.randint(100, 5000)
            },
            "status": "OK" if random.random() > 0.05 else "WARNING"
        })
    return telemetry

print("Generator functions defined successfully.")

## 5. Execute Bulk Ingestion

In [ ]:
def batch_insert(collection, docs: List[Dict[str, Any]], batch_size: int = 1000):
    total = len(docs)
    inserted_count = 0
    start_time = time.time()
    
    for i in range(0, total, batch_size):
        chunk = docs[i:i + batch_size]
        res = collection.insert_many(chunk)
        inserted_count += len(res.inserted_ids)
        
    elapsed = time.time() - start_time
    rate = inserted_count / elapsed if elapsed > 0 else 0
    print(f"  ✓ Collection '{collection.name}': Inserted {inserted_count:,} docs in {elapsed:.2f}s ({rate:,.1f} docs/sec)")

# 1. Customers
print(f"Generating and inserting {NUM_CUSTOMERS:,} customer records...")
customer_docs = generate_customers(NUM_CUSTOMERS)
batch_insert(db["customers"], customer_docs, BATCH_SIZE)

# 2. Products
print(f"\nGenerating and inserting {NUM_PRODUCTS:,} product records...")
product_docs = generate_products(NUM_PRODUCTS)
batch_insert(db["products"], product_docs, BATCH_SIZE)

# 3. Orders
customer_ids = [c["customer_id"] for c in customer_docs]
print(f"\nGenerating and inserting {NUM_ORDERS:,} order records...")
order_docs = generate_orders(NUM_ORDERS, customer_ids, product_docs)
batch_insert(db["orders"], order_docs, BATCH_SIZE)

# 4. Sensor Telemetry
print(f"\nGenerating and inserting {NUM_SENSOR_LOGS:,} sensor telemetry records...")
sensor_docs = generate_sensor_telemetry(NUM_SENSOR_LOGS)
batch_insert(db["sensor_telemetry"], sensor_docs, BATCH_SIZE)

print("\n🎉 All synthetic dataset collections successfully populated!")

## 6. Build Database Indexes

In [ ]:
print("Building collection indexes...")

# Customers indexes
db["customers"].create_index("customer_id", unique=True)
db["customers"].create_index("email")
db["customers"].create_index([("address.state", 1), ("account_tier", 1)])

# Products indexes
db["products"].create_index("product_id", unique=True)
db["products"].create_index("category")
db["products"].create_index("price")

# Orders indexes
db["orders"].create_index("order_id", unique=True)
db["orders"].create_index("customer_id")
db["orders"].create_index("order_date")
db["orders"].create_index([("status", 1), ("order_date", -1)])

# Sensor Telemetry indexes
db["sensor_telemetry"].create_index([("device_id", 1), ("timestamp", -1)])
db["sensor_telemetry"].create_index("location")

print("✅ Collection indexes generated successfully.")

## 7. Cluster Verification & Analytical Aggregations

In [ ]:
print("=================== Database Summary ===================")
for col_name in ["customers", "products", "orders", "sensor_telemetry"]:
    count = db[col_name].count_documents({})
    stats = db.command("collStats", col_name)
    size_mb = stats.get("size", 0) / (1024 * 1024)
    storage_mb = stats.get("storageSize", 0) / (1024 * 1024)
    indexes = len(stats.get("indexSizes", {}))
    print(f"Collection '{col_name:18s}': {count:7,d} docs | Size: {size_mb:5.2f} MB | Storage: {storage_mb:5.2f} MB | Indexes: {indexes}")

print("\n=================== Top Revenue Products ===================")
pipeline = [
    {"$unwind": "$items"},
    {"$group": {
        "_id": "$items.product_name",
        "total_units_sold": {"$sum": "$items.quantity"},
        "total_revenue": {"$sum": "$items.subtotal"}
    }},
    {"$sort": {"total_revenue": -1}},
    {"$limit": 5}
]

for top in db["orders"].aggregate(pipeline):
    print(f"Product: {top['_id']:35s} | Units: {top['total_units_sold']:4d} | Revenue: ${top['total_revenue']:10.2f}")

print("\n=================== Telemetry by Zone ===================")
telemetry_pipeline = [
    {"$group": {
        "_id": "$location",
        "avg_cpu": {"$avg": "$metrics.cpu_utilization_pct"},
        "avg_temp": {"$avg": "$metrics.temperature_celsius"},
        "total_events": {"$sum": 1}
    }},
    {"$sort": {"_id": 1}}
]

for loc in db["sensor_telemetry"].aggregate(telemetry_pipeline):
    print(f"Zone: {loc['_id']:15s} | Avg CPU: {loc['avg_cpu']:5.2f}% | Avg Temp: {loc['avg_temp']:5.1f}°C | Events: {loc['total_events']:,}")